# Preprocessing on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on removing outliers in HVFHV dataset and drop unused columns.

----

# Import Libraries:

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import min as spark_min, max as spark_max, col
from pyspark.sql.functions import * 
import os

In [4]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.executor.heartbeatInterval", "30s")
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/22 03:16:38 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/22 03:16:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/22 03:16:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/22 03:16:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Read HVFHV Parquet Files:

In [5]:
base_dir = "../data"
hvfhv_path = base_dir + '/raw/hvfhv_data/'
hvfhv_sdf = spark.read.parquet(hvfhv_path)

In [4]:
hvfhv_sdf.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [4]:
# Check the shape of parquet file
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 117277281
Number of columns: 24


In [6]:
hvfhv_sdf.show(5)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

Change the unit of `trip_time` from seconds to minutes:

In [4]:
hvfhv_sdf = hvfhv_sdf.withColumn('trip_time', F.round(F.col('trip_time') / 60, 2))
hvfhv_sdf.show(5)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

# Drop Unrelated Rows and Columns:

In [6]:
# Calculate the amount of NULL in each column
na_counts = hvfhv_sdf.select([sum(col(column).isNull().cast("int")).alias(column) for column in hvfhv_sdf.columns])
na_counts.show()

+-----------------+--------------------+--------------------+----------------+-----------------+---------------+----------------+------------+------------+----------+---------+-------------------+-----+---+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|request_datetime|on_scene_datetime|pickup_datetime|dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls|bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+----------------+-----------------+---------------+----------------+------------+------------+----------+---------+-------------------+-----+---+---------+--------------------+-----------+----+----------+-------------------+-----

Since imputing values in `originating_base_num` and `on_scene_datetime` are illogical, we drop these two columns.

In [5]:
hvfhv_sdf = hvfhv_sdf.drop("originating_base_num", "on_scene_datetime")
hvfhv_sdf.show(5)

+-----------------+--------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|   request_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|           HV0003|

In [7]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 117277281
Number of columns: 22


In [6]:
numeric_columns = ['trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax',
                  'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay']

# Outlier Detection:

### Find the earliest and latest of datetime variables:

In [10]:
# Time period for `request_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("request_datetime")).alias("min_date"),
    spark_max(col("request_datetime")).alias("max_date")
    ).collect()

print("request_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `pickup_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("pickup_datetime")).alias("min_date"),
    spark_max(col("pickup_datetime")).alias("max_date")
    ).collect()

print("pickup_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `dropoff_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("dropoff_datetime")).alias("min_date"),
    spark_max(col("dropoff_datetime")).alias("max_date")
    ).collect()

print("dropoff_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

request_datetime:
Min: 2023-06-30 21:39:45
Max: 2024-01-01 00:15:00


pickup_datetime:
Min: 2023-07-01 00:00:00
Max: 2023-12-31 23:59:59


dropoff_datetime:
Min: 2023-07-01 00:02:33
Max: 2024-01-01 06:04:47


Remove the rows when `request_datetime` starts with "2023-06-30" and "2024-01-01" and `dropoff_datetime` starts with "2024-01-01":

In [8]:
hvfhv_sdf = hvfhv_sdf.filter(~(F.col('request_datetime').startswith('2023-06-30')) &
                              ~ (F.col('request_datetime').startswith('2024-01-01')) & 
                              ~(F.col('dropoff_datetime').startswith('2024-01-01')))

### Calculate the descriptive statistics of numerical columns:

In [10]:
hvfhv_sdf.select(numeric_columns).describe().show()

+-------+------------------+------------------+-------------------+------------------+------------------+-----------------+--------------------+-------------------+------------------+------------------+
|summary|        trip_miles|         trip_time|base_passenger_fare|             tolls|               bcf|        sales_tax|congestion_surcharge|        airport_fee|              tips|        driver_pay|
+-------+------------------+------------------+-------------------+------------------+------------------+-----------------+--------------------+-------------------+------------------+------------------+
|  count|         117268207|         117268207|          117268207|         117268207|         117268207|        117268207|           117268207|          117268207|         117268207|         117268207|
|   mean| 5.130569780767707| 20.33986603743354| 25.368622719036882|1.1703865681256698|0.7319522602596277|2.125379932590766|  1.0797475376254366|0.22133557955738162|1.2017110650461427|20.03

### Filter out the unrealistic data:
- Remove the data in `PULocationID` and `DOLocationID` which out of the range
- Assume the `trip_miles` is greater than 1 mile, `trip_time` is greater than 1 minute, `base_passenger_fare` is greater than 1 dollar
- Assume the maximum `trip_time` is 300 minutes
- Assume `driver_pay` is greater than 0

In [9]:
hvfhv_sdf = hvfhv_sdf.filter((hvfhv_sdf.PULocationID >= 1) & 
                             (hvfhv_sdf.PULocationID <= 263) &
                             (hvfhv_sdf.DOLocationID >= 1) & 
                             (hvfhv_sdf.DOLocationID <= 263) &
                             (hvfhv_sdf.trip_miles > 1) &
                             (hvfhv_sdf.trip_time > 1) & 
                             (hvfhv_sdf.trip_time <= 300) &
                             (hvfhv_sdf.base_passenger_fare > 1) &
                             (hvfhv_sdf.driver_pay > 0))

num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 22


### Calculate the descriptive statistics of numerical columns again:

In [11]:
hvfhv_sdf.select(numeric_columns).describe().show()

+-------+-----------------+------------------+-------------------+------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+
|summary|       trip_miles|         trip_time|base_passenger_fare|             tolls|               bcf|         sales_tax|congestion_surcharge|        airport_fee|              tips|        driver_pay|
+-------+-----------------+------------------+-------------------+------------------+------------------+------------------+--------------------+-------------------+------------------+------------------+
|  count|        100725779|         100725779|          100725779|         100725779|         100725779|         100725779|           100725779|          100725779|         100725779|         100725779|
|   mean|5.034872710788129| 21.02144771081905| 25.085769118951625|0.8429820415687126|0.7156277896849682|2.2333404546740576|  1.1387507536675392|0.23961212054761075|1.1501649327528747|19.98

### Check the time period of the dataset again:

In [10]:
# Time period for `request_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("request_datetime")).alias("min_date"),
    spark_max(col("request_datetime")).alias("max_date")
    ).collect()

print("request_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `pickup_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("pickup_datetime")).alias("min_date"),
    spark_max(col("pickup_datetime")).alias("max_date")
    ).collect()

print("pickup_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `dropoff_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("dropoff_datetime")).alias("min_date"),
    spark_max(col("dropoff_datetime")).alias("max_date")
    ).collect()

print("dropoff_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

request_datetime:
Min: 2023-07-01 00:00:00
Max: 2023-12-31 23:55:17


pickup_datetime:
Min: 2023-07-01 00:00:46
Max: 2023-12-31 23:56:02


dropoff_datetime:
Min: 2023-07-01 00:06:45
Max: 2023-12-31 23:59:59


Now, the HVFHV dataset is in the correct time period and its outliers have been removed.

# Save the Preprocessed HVFHV Dataset:

In [13]:
hvfhv_dir = base_dir + '/curated/hvfhv_data'
file_name = 'preprocessed_hvfhv'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)